# Quantitative Signal Discovery Agent — Brev Launchable Setup

[![Click here to deploy.](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/)

This launchable introduces NVIDIA's **Quantitative Signal Discovery Agent** developer blueprint. It uses the [NeMo Agent Toolkit](https://docs.nvidia.com/nemo-agent-toolkit/) to orchestrate three LLM-powered agents that automatically generate, code, and evaluate alpha signals for quantitative finance.

In particular, the main notebook will cover the following:
- **Signal Discovery Setup** – Clone the repository, install dependencies, and configure the environment
- **Signal Generation** – Use an LLM to propose signal expressions over price/volume operators
- **Code Generation** – Translate signal formulas into self-contained, executable Python
- **Backtesting on S&P 500 Data** – Compute Rank IC and statistical significance
- **Closed-Loop Optimization** – Iterate with advisor-driven feedback until the signal meets the IC threshold
- **Tracing with Phoenix** – Inspect every LLM call, token count, and latency end-to-end

In this **setup notebook**, we focus on the essential steps required to prepare the environment and install the dependencies. Once setup is complete, you will jump to the main notebook, [signal-discovery-workflow.ipynb](../notebooks/signal-discovery-workflow.ipynb).

---

<a id="environment-setup"></a>
## Environment Setup

Run the cells below to clone the repo, configure the project path, install dependencies, and register the Jupyter kernel.

**Note:** only clone the GitHub repository when deploying the Brev launchable. If you are running this notebook from inside an already-cloned checkout, skip the `git clone` cell.

In [ ]:
!git clone https://github.com/NVIDIA-AI-Blueprints/quantitative-signal-discovery-agent.git

In [ ]:
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()

candidates = [
    notebook_dir / "quantitative-signal-discovery-agent",
    notebook_dir.parent,
    notebook_dir,
]
project_root = next((p for p in candidates if (p / "pyproject.toml").exists()), None)
if project_root is None:
    raise FileNotFoundError(
        "Could not locate the quantitative-signal-discovery-agent checkout. "
        "Run the `git clone` cell above first, or open this notebook from inside the repo."
    )

local_bin = os.path.expanduser("~/.local/bin")
os.environ["PATH"] = f"{local_bin}:{os.environ['PATH']}"
print(f"Updated PATH to include: {local_bin}")

sys.path.insert(0, str(project_root / "src"))
os.chdir(project_root)
print(f"Working directory changed to: {Path.cwd()}")

### Install `uv` and Project Dependencies

The repo is managed with [`uv`](https://docs.astral.sh/uv/), a fast Python package manager. The cell below installs `uv` if missing, creates the project virtual environment, and registers a dedicated Jupyter kernel named **Signal Discovery**.

In [ ]:
import subprocess

subprocess.run("curl -sSf https://astral.sh/uv/install.sh | sh", shell=True, check=True)

# Register the kernel with the venv's bin/ AND ~/.local/bin (where `uv`
# itself lives after `curl ... | sh`) baked into PATH. Without this,
# `subprocess.Popen(["phoenix", ...])` and `subprocess.run(["uv", ...])`
# don't resolve once the user switches kernels.
install_script = """
unset VIRTUAL_ENV
export PATH="$HOME/.local/bin:$PATH"
uv venv
uv pip install -e ".[test]"
VENV_BIN="$(uv run python -c 'import sys, os; print(os.path.dirname(sys.executable))')"
uv run python -m ipykernel install --user --name=signal-discovery \\
    --display-name "Signal Discovery" \\
    --env PATH "${VENV_BIN}:${HOME}/.local/bin:${PATH}"
"""

subprocess.run(install_script, shell=True, executable="/bin/bash", check=True)
print("\n\u2705 Environment ready and 'Signal Discovery' kernel registered with venv bin/ on PATH.")

### Configure Your NVIDIA API Key

The workflow calls hosted NVIDIA NIM endpoints for the Signal, Code, and Advisor agents. Get a key from [build.nvidia.com](https://build.nvidia.com/settings/api-keys) and paste it below.

The key is written to a `.env` file at the project root so the main notebook can pick it up automatically; it is **not** committed to git (the repo's `.gitignore` covers it).

In [1]:
import getpass
import os
import subprocess
from pathlib import Path

env_path = Path(".env")

def _load_env_file(path: Path) -> dict[str, str]:
    """Parse a simple KEY=VALUE .env file. Returns {} if the file is missing."""
    if not path.exists():
        return {}
    out = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        out[k.strip()] = v.strip().strip('"').strip("'")
    return out

env_file = _load_env_file(env_path)

api_key = (
    os.environ.get("NVIDIA_API_KEY")
    or env_file.get("NVIDIA_API_KEY")
    or getpass.getpass("Enter your NVIDIA_API_KEY: ").strip()
)
if not api_key:
    raise ValueError("NVIDIA_API_KEY is required to run the workflow.")

os.environ["NVIDIA_API_KEY"] = api_key

env_file["NVIDIA_API_KEY"] = api_key
env_path.write_text("\n".join(f"{k}={v}" for k, v in env_file.items()) + "\n")
print(f"\u2705 NVIDIA_API_KEY set in environment and persisted to {env_path.resolve()}")
print("   (.env is gitignored — safe to keep on disk.)")

✅ NVIDIA_API_KEY set in environment and persisted to /Users/phuo/Desktop/quant-factor-mining-agent/brev/.env
   (.env is gitignored — safe to keep on disk.)


### Verify Installation

After the install cell finishes, **select the `Signal Discovery` kernel** from the kernel menu (top-right of Jupyter), then run the cell below to confirm every required package imports cleanly.

In [ ]:
import importlib
import shutil

packages = [
    "numpy", "pandas", "scipy", "yfinance",
    "nat", "phoenix",
    "signal_discovery_workflow",
]
executables = ["phoenix", "nat"]

print("Checking packages...\n")
failed = []
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "installed")
        print(f"\u2713 {pkg:30s} {ver}")
    except ImportError:
        print(f"\u2717 {pkg:30s} FAILED")
        failed.append(pkg)

print("\nChecking console scripts on PATH...\n")
missing_exes = []
for exe in executables:
    path = shutil.which(exe)
    if path:
        print(f"\u2713 {exe:30s} {path}")
    else:
        print(f"\u2717 {exe:30s} NOT FOUND on PATH")
        missing_exes.append(exe)

problems = []
if failed:
    problems.append('Missing packages: ' + ', '.join(failed))
if missing_exes:
    problems.append(
        f"Console scripts not on PATH: {', '.join(missing_exes)}. "
        "Re-run the install cell above so the kernel is registered with the venv's bin/ on PATH."
    )
summary = ('\u274c ' + ' | '.join(problems)) if problems else '\u2705 All packages and console scripts ready!'
print(f"\n{summary}")

---
## Setup Complete!

If you see **✅ All packages ready!** above, the Python environment, kernel, API key, and dataset are all in place.

You can now jump to the main walkthrough:

➡️ **[Open the Signal Discovery Workflow Notebook](./quantitative-signal-discovery-agent/notebooks/signal-discovery-workflow.ipynb)** ⬅️

Make sure to switch the kernel of that notebook to **Signal Discovery** before running its cells.

---

SPDX-FileCopyrightText: Copyright (c) 2023-2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.

SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0. Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.